In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import  OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [10]:
df = pd.read_csv('train.csv')

In [11]:
X = df.drop(columns=['id', 'accident_risk'])
y = df['accident_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

categorical_features = ['road_type', 'lighting', 'weather', 'time_of_day']
numerical_features = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
bool_features = ['road_signs_present', 'public_road', 'holiday', 'school_season']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features + bool_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [ ]:
numerical_features = ['curvature', 'speed_limit', 'num_reported_accidents']
categorical_features = ['lighting', 'weather']
boolean_features = ['public_road', 'holiday']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('bool', 'passthrough', boolean_features)
    ],
    remainder='drop'
)

   id  curvature  speed_limit  lighting weather  public_road  holiday  \
0   0  -1.572918    -0.703840  daylight   rainy         True    False   
1   1   1.839137    -0.703840  daylight   clear        False     True   
2   2   0.518342     1.512963       dim   clear         True     True   
3   3  -1.536229    -0.703840       dim   rainy         True    False   
4   4   0.334898     0.879591  daylight   foggy        False     True   

   num_reported_accidents  accident_risk  
0               -0.209797           0.13  
1               -1.325918           0.35  
2                0.906324           0.30  
3               -0.209797           0.21  
4               -0.209797           0.56  


In [13]:
Enchanted_forest = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,   
        max_depth=10,       
        random_state=42     
    ))
])

Enchanted_forest.fit(X_train, y_train)

predictions = Enchanted_forest.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, predictions))

print(f"Random Forest RMSE: {rmse:.4f}")

Random Forest RMSE: 0.0555


## **Enchanted forest RMSE 0.0556**

In [14]:
final_df = pd.read_csv('test.csv')

Enchanted_forest_predictions = Enchanted_forest.predict(final_df.drop(columns=['id']))

submission = pd.DataFrame({
    'id': final_df['id'],
    'accident_risk': Enchanted_forest_predictions
})

submission.to_csv('submission_enchanted_forest.csv', index=False)

## **Fine-tune the model**

In [19]:
# base_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(random_state=42))
# ])

# param_grid = {
#     'regressor__n_estimators': [100, 200, 300],
#     'regressor__max_depth': [None, 10, 20],
#     'regressor__min_samples_split': [2, 5, 10],
#     'regressor__max_features': ['sqrt', 'log2']
# }

# grid_search = GridSearchCV(
#     estimator=base_pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring='neg_root_mean_squared_error',
#     n_jobs=-1
# )

# grid_search.fit(X_train, y_train)

# Enchanted_forest_finetuned = grid_search.best_estimator_
# y_pred = Enchanted_forest_finetuned.predict(X_test)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# print(f"Best Parameters: {grid_search.best_params_}")
# print(f"Final RMSE: {rmse:.4f}")